# Prepare Re-Verification — Flat Folder\n\nReads all JSON files directly from `dataset_ds_to_verify/` (no subfolders)  \nand writes a new flat folder `dataset_ds_reverif/`, keeping only:\n- `id`\n- `specie`\n- `specie_en`\n- `corrected_text`  ← from `verification.corrected_text`, fallback to `generated_text`

In [1]:
import json
from pathlib import Path

In [2]:
input_dir  = Path("../dataset_ds_to_verify")
output_dir = Path("../dataset_gpt_4_reverif")
output_dir.mkdir(exist_ok=True)

json_files = sorted(input_dir.glob("*.json"))
print(f"Found {len(json_files)} JSON files in '{input_dir}'")

Found 129 JSON files in '../dataset_ds_to_verify'


In [3]:
copied         = 0
skipped        = 0
missing_fields = []

for src in json_files:
    data = json.loads(src.read_text(encoding="utf-8"))

    # corrected_text is a top-level key in these files (not nested under verification)
    corrected_text = data.get("corrected_text")

    if corrected_text is None:
        missing_fields.append(str(src))
        skipped += 1
        continue

    record = {
        "id":             data.get("id"),
        "specie":         data.get("specie"),
        "corrected_text": corrected_text,
    }

    dst = output_dir / src.name
    dst.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    copied += 1

print(f"Done: {copied} files written, {skipped} skipped (missing corrected_text)")
print(f"Output folder: {output_dir.resolve()}")

if missing_fields:
    print("\nFiles with missing corrected_text:")
    for f in missing_fields:
        print(f"  {f}")

Done: 129 files written, 0 skipped (missing corrected_text)
Output folder: /Users/somaya/Documents/Phd/github_version/version_2/deb-data-pipeline-github/data_verification/dataset_gpt_4_reverif


In [4]:
# Fallback: for files missing corrected_text, use generated_text instead
fallback_copied  = 0
fallback_skipped = 0
still_missing    = []

for src_path in missing_fields:
    src  = Path(src_path)
    data = json.loads(src.read_text(encoding="utf-8"))

    generated_text = data.get("generated_text")

    if generated_text is None:
        still_missing.append(src_path)
        fallback_skipped += 1
        continue

    record = {
        "id":             data.get("id"),
        "specie":         data.get("specie"),
        "corrected_text": generated_text,   # fallback to generated_text
    }

    dst = output_dir / src.name
    dst.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    fallback_copied += 1

print(f"Fallback: {fallback_copied} files written using generated_text, {fallback_skipped} still skipped")
if still_missing:
    print("\nFiles with neither corrected_text nor generated_text:")
    for f in still_missing:
        print(f"  {f}")

Fallback: 0 files written using generated_text, 0 still skipped


In [5]:
# Sanity check: compare total file counts
src_count = len(list(input_dir.glob("*.json")))
out_count = len(list(output_dir.glob("*.json")))
match     = "OK" if src_count == out_count else "MISMATCH"

print(f"Source : {src_count} files")
print(f"Output : {out_count} files")
print(f"Status : {match}")

Source : 129 files
Output : 129 files
Status : OK
